# Error Analysis: Low-Score Samples

Analyze samples with low judge scores (Coverage ≤ 2, Specificity ≤ 2, or Consistency ≤ 2)

In [17]:
import json
from pathlib import Path
from IPython.display import display, HTML

## Load Low-Score Samples

In [18]:
# Load samples
samples = []
file_path = Path("../../eval/results/low_score_samples.jsonl")

if file_path.exists():
    with open(file_path) as f:
        for line in f:
            samples.append(json.loads(line))
    print(f"Loaded {len(samples)} low-score samples")
else:
    print(f"File not found: {file_path}")
    print("Run eval_and_analysis.py first to generate low-score samples")

Loaded 9 low-score samples


## Summary Statistics

In [19]:
if samples:
    # Count by issue type
    low_coverage = sum(1 for s in samples if s['scores']['coverage'] <= 2)
    low_specificity = sum(1 for s in samples if s['scores']['specificity'] <= 2)
    low_consistency = sum(1 for s in samples if s['scores']['consistency'] <= 2)
    
    print("Issue Distribution:")
    print(f"  Low Coverage (≤2): {low_coverage}")
    print(f"  Low Specificity (≤2): {low_specificity}")
    print(f"  Low Consistency (≤2): {low_consistency}")
    print(f"\nNote: Some samples may have multiple issues")
    
    # Average scores
    avg_coverage = sum(s['scores']['coverage'] for s in samples) / len(samples)
    avg_specificity = sum(s['scores']['specificity'] for s in samples) / len(samples)
    avg_consistency = sum(s['scores']['consistency'] for s in samples) / len(samples)
    avg_conciseness = sum(s['scores']['conciseness'] for s in samples) / len(samples)
    
    print(f"\nAverage Scores (low-score samples only):")
    print(f"  Coverage: {avg_coverage:.2f}/5")
    print(f"  Specificity: {avg_specificity:.2f}/5")
    print(f"  Consistency: {avg_consistency:.2f}/5")
    print(f"  Conciseness: {avg_conciseness:.2f}/5")

Issue Distribution:
  Low Coverage (≤2): 0
  Low Specificity (≤2): 8
  Low Consistency (≤2): 2

Note: Some samples may have multiple issues

Average Scores (low-score samples only):
  Coverage: 3.22/5
  Specificity: 2.11/5
  Consistency: 4.22/5
  Conciseness: 3.33/5


## Side-by-Side Comparison

Display document, reference, and model output for each low-score sample

In [20]:
def display_sample(sample):
    """Display a sample with side-by-side comparison."""
    doc = sample['document']
    ref = sample['reference']
    gen = sample['base_summary']
    scores = sample['scores']
    
    # Color-code scores
    def score_color(score):
        if score <= 2:
            return 'red'
        elif score <= 3:
            return 'orange'
        else:
            return 'green'
    
    html = f"""
    <div style="border: 2px solid #333; padding: 15px; margin: 20px 0; background-color: #f9f9f9;">
        <h3>Sample {sample['sample_id']} - Category: {sample['category']}</h3>
        
        <div style="margin: 10px 0;">
            <strong>Scores:</strong>
            <span style="color: {score_color(scores['coverage'])};">Coverage: {scores['coverage']}/5</span> | 
            <span style="color: {score_color(scores['specificity'])};">Specificity: {scores['specificity']}/5</span> | 
            <span style="color: {score_color(scores['consistency'])};">Consistency: {scores['consistency']}/5</span> | 
            <span style="color: {score_color(scores['conciseness'])};">Conciseness: {scores['conciseness']}/5</span>
        </div>
        
        <hr>
        
        <div style="margin: 15px 0;">
            <h4 style="background-color: #e0e0e0; padding: 8px;">📄 Full Document ({len(doc)} chars)</h4>
            <pre style="white-space: pre-wrap; background-color: white; padding: 10px; border: 1px solid #ccc; max-height: 400px; overflow-y: auto;">{doc}</pre>
        </div>
        
        <div style="margin: 15px 0;">
            <h4 style="background-color: #d4edda; padding: 8px;">✅ Reference Summary ({len(ref.split())} words)</h4>
            <pre style="white-space: pre-wrap; background-color: white; padding: 10px; border: 1px solid #28a745;">{ref}</pre>
        </div>
        
        <div style="margin: 15px 0;">
            <h4 style="background-color: #fff3cd; padding: 8px;">🤖 Model Output ({len(gen.split())} words)</h4>
            <pre style="white-space: pre-wrap; background-color: white; padding: 10px; border: 1px solid #ffc107;">{gen}</pre>
        </div>
        
        <div style="margin: 15px 0; background-color: #f8d7da; padding: 10px; border: 1px solid #f5c6cb;">
            <h4>🔍 Judge Explanation</h4>
            <p>{sample['explanation']}</p>
        </div>
    </div>
    """
    
    display(HTML(html))

### Display All Low-Score Samples

In [21]:
if samples:
    for sample in samples:
        display_sample(sample)
else:
    print("No samples to display")

## Filter by Specific Issue

In [22]:
# Filter samples by issue type
issue_type = 'consistency'  # Change to 'coverage', 'specificity', or 'consistency'

filtered = [s for s in samples if s['scores'][issue_type] <= 2]
print(f"\nShowing {len(filtered)} samples with low {issue_type}:\n")

for sample in filtered:
    display_sample(sample)


Showing 2 samples with low consistency:



## Export Worst Samples

Find samples with multiple low scores

In [23]:
if samples:
    # Find samples with 2+ dimensions scoring ≤2
    worst_samples = []
    for s in samples:
        low_count = sum([
            s['scores']['coverage'] <= 2,
            s['scores']['specificity'] <= 2,
            s['scores']['consistency'] <= 2
        ])
        if low_count >= 2:
            worst_samples.append(s)
    
    print(f"Found {len(worst_samples)} samples with 2+ low scores:\n")
    
    for sample in worst_samples:
        display_sample(sample)

Found 1 samples with 2+ low scores:

